In [74]:
from dotenv import load_dotenv
load_dotenv()

True

In [75]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate


In [76]:
import langchain
print(langchain.__version__)

1.2.15


In [77]:
loader = PyPDFLoader("..\\data\\medical_report.pdf")
docs = loader.load()
len(docs)

9

In [78]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

26

In [79]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [80]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding= embeddings
)

In [81]:
# query = "machine learning and Data Science content"

In [82]:
# data = vector_store.similarity_search(query=query)

In [83]:
# data[0]
# context=""
# for doc in data:
#     context += doc.page_content + "\n"



In [84]:
llm = ChatOpenAI(model="gpt-5")

In [85]:
# res = llm.invoke(f""" Can you provide me the answer based on provided context for my question, context: {context} and question: {query}""")
# print(res)

In [86]:
def get_context(query):
    data = vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context += doc.page_content + "\n"
    return {
        "context": {context},
        "question": {query}
    }

In [87]:
prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Provide answers based only on the given context.

If the answer is not present in the context, say "I don't know." Do not make up an answer.

Context:
{context}

Question:
{question}
"""
)

In [88]:
rag_chain = get_context| prompt| llm

In [90]:
res = rag_chain.invoke("the wbc count is ok or not?")
print(res.content)

Yes. Your total white blood cell (TLC) count is 7.33 thou/mm3, which is within the reference range of 4.00–10.00.

Note: In the differential, lymphocytes are slightly above the stated ranges: 41.9% (ref 20–40%) and 3.07 thou/mm3 (ref 1.00–3.00). All other differential counts are within their reference ranges.
